# **Neural Networks and Differential Equations: From Infinite Layers to Continuous Modelling**

## **Part III: Neural ODEs**

### **Tutorial @ IJCAI 2026**

#### **Cecília Coelho, Luís Ferrás and Andrzej Dulny**

#### **Go to [website][Website]**

[Website]: https://ceciliacoelho.github.io/tutorialNN4DEs

# --------------------------------------------------------

In [ ]:
pip install torchdiffeq

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchdiffeq import odeint

### Learning a population dynamics

In [ ]:
#########################################Building the synthetic dataset
device = torch.device('cuda:' + str(args.gpu) if torch.cuda.is_available() else 'cpu')

true_y0 = torch.tensor([2.518629]).to(device)

t = torch.linspace(0., 1, 500).to(device)
t_test = torch.linspace(0., 1, 700).to(device)



class Lambda(nn.Module):

    def forward(self, t, y):
        return torch.mul(torch.mul(0.026, y), torch.sub(1, torch.div(y,12)))


with torch.no_grad():
    true_y = odeint(Lambda(), true_y0, t, method='dopri5')
    test_y = odeint(Lambda(), true_y0, t_test, method='dopri5')
########################################

class ODEFunc(nn.Module):

    def __init__(self):
        super(ODEFunc, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(1, 50),
            nn.Tanh(),
            nn.Linear(50,50),
            nn.ELU(),
            nn.Linear(50, 1),
        )

        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, val=0)

    def forward(self, t, y):
        return self.net(y)

if __name__ == '__main__':

    func = ODEFunc().to(device)

    optimizer = optim.Adam(func.parameters(), lr=1e-5)

    for itr in range(1, 2001):
        pred_y = odeint(func, true_y0, t, method='rk4').to(device)
        loss = nn.MSELoss()(pred_y, true_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        if itr % 1 == 0:
            with torch.no_grad():
                print('Iter {:04d} | MSE Loss {:.6f}'.format(itr, loss.item()))

                if itr == 2000:
                    pred_y_test = odeint(func, true_y0, t_test)
                    mse_t = nn.MSELoss()(pred_y_test, test_y)
                    print('MSE Loss {:.6f}'.format(mse_t.item()))
                    plt.plot(t_test.detach().cpu().numpy(), test_y.detach().cpu().numpy(), linestyle='dashed', label='real')
                    plt.plot(t_test.detach().cpu().numpy(), pred_y_test.detach().cpu().numpy(), label='predicted')
                    plt.xlabel("Time")
                    plt.ylabel("Population")
                    plt.legend()
                    plt.show()


Iter 0001 | MSE Loss 0.000784
Iter 0002 | MSE Loss 0.000770
Iter 0003 | MSE Loss 0.000755
Iter 0004 | MSE Loss 0.000741
Iter 0005 | MSE Loss 0.000727
Iter 0006 | MSE Loss 0.000713
Iter 0007 | MSE Loss 0.000699
Iter 0008 | MSE Loss 0.000686
Iter 0009 | MSE Loss 0.000672
Iter 0010 | MSE Loss 0.000659
Iter 0011 | MSE Loss 0.000646
Iter 0012 | MSE Loss 0.000633
Iter 0013 | MSE Loss 0.000620
Iter 0014 | MSE Loss 0.000607
Iter 0015 | MSE Loss 0.000595
Iter 0016 | MSE Loss 0.000582
Iter 0017 | MSE Loss 0.000570
Iter 0018 | MSE Loss 0.000558
Iter 0019 | MSE Loss 0.000546
Iter 0020 | MSE Loss 0.000535
Iter 0021 | MSE Loss 0.000523
Iter 0022 | MSE Loss 0.000512
Iter 0023 | MSE Loss 0.000501
Iter 0024 | MSE Loss 0.000490
Iter 0025 | MSE Loss 0.000479
Iter 0026 | MSE Loss 0.000468
Iter 0027 | MSE Loss 0.000458
Iter 0028 | MSE Loss 0.000447
Iter 0029 | MSE Loss 0.000437
Iter 0030 | MSE Loss 0.000427
Iter 0031 | MSE Loss 0.000417
Iter 0032 | MSE Loss 0.000408
Iter 0033 | MSE Loss 0.000398
Iter 0034 